# Week 02 Vector Database Basics

이 노트는 제공받은 PDF `Part2.pdf`의 **Part 4: Vector Database Basics**를 바탕으로, RAG 초심자가 이해하기 쉬운 흐름으로 다시 정리한 2주차 학습 자료입니다.

## 이번 주 목표

- Vector Index와 Vector Database의 차이를 설명할 수 있다.
- Indexing과 Querying이 왜 나뉘는지 이해한다.
- ANN, IVF, HNSW, Quantization이 왜 등장하는지 큰 흐름을 이해한다.
- Dense, Sparse, Hybrid Search가 각각 언제 필요한지 말할 수 있다.
- Vector DB 제품을 비교할 때 어떤 기준을 봐야 하는지 감을 잡는다.


## 어떻게 읽으면 좋은가

1. 먼저 `왜 Vector Database가 필요한가`부터 읽고 전체 맥락을 잡습니다.
2. 그다음 `Indexing` 파트에서 검색 속도를 높이는 원리를 봅니다.
3. 이후 `Querying` 파트에서 Dense, Sparse, Hybrid Search의 차이를 이해합니다.
4. 마지막으로 제품 비교와 체크리스트를 보며 실무 관점으로 연결합니다.

이 노트는 코드를 많이 작성하는 주차가 아니라, **RAG 검색 인프라의 구조를 이해하는 주차**입니다.


## 왜 2주차에 Vector Database를 공부할까

1주차에서 RAG, 파인튜닝, 데이터 플로우, 청킹까지 공부했다면 다음 질문이 자연스럽게 나옵니다.

- 청킹한 문서를 어디에 저장하지?
- 질문이 들어왔을 때 관련 청크를 어떻게 빨리 찾지?
- 문서가 수천 개, 수십만 개가 되면 어떻게 검색하지?

이 질문에 답하는 영역이 바로 **Vector Database**입니다.

RAG에서 문서는 보통 다음 흐름으로 처리됩니다.

```text
문서 -> 청킹 -> 임베딩 -> 벡터 저장 -> 검색 -> 프롬프트 구성 -> LLM 응답
```

즉, 2주차는 `청킹 이후 검색 전까지`의 핵심 구간을 이해하는 주차라고 보면 됩니다.


## PDF에서 확인되는 큰 목차 흐름

제공된 PDF에서는 대체로 아래 순서로 내용이 전개됩니다.

- `Vector Database Basics - Overview`
- `Indexing`
- `Querying`
- `Vector Database`
- `Pinecone Indexing & Querying`

이 노트도 그 흐름을 따라가되, 초심자 입장에서 이해하기 쉬운 설명을 추가했습니다.


## 핵심 키워드

- `Vector`
- `Embedding`
- `Vector Index`
- `Vector Database`
- `ANN`
- `kNN`
- `Indexing`
- `Querying`
- `IVF`
- `HNSW`
- `Quantization`
- `Sparse Search`
- `Dense Search`
- `Hybrid Search`
- `Metadata Filtering`
- `RRF`


## Vector Database란 무엇인가

Vector Database는 임베딩 벡터를 저장하고, 비슷한 벡터를 빠르게 찾기 위한 시스템입니다.

RAG에서 LLM은 문서를 직접 기억하지 않습니다. 대신 질문과 비슷한 의미를 가진 문서 조각을 먼저 찾아서 프롬프트에 붙입니다. 이때 필요한 것이 벡터 저장과 검색 시스템입니다.

쉽게 말하면:

- 청킹은 문서를 검색 가능한 단위로 나누는 일
- 임베딩은 각 청크를 숫자 벡터로 바꾸는 일
- Vector Database는 그 벡터를 저장하고 다시 찾는 일

즉, Vector Database는 RAG의 `검색 엔진 쪽 뇌`에 가깝습니다.


## Vector Index vs Vector Database

| 항목 | Vector Index | Vector Database |
|---|---|---|
| 핵심 역할 | 벡터를 빠르게 찾기 위한 검색 구조 | 벡터 저장, 검색, 필터링, 운영 기능을 포함한 시스템 |
| 관심사 | 검색 속도와 근사 탐색 | CRUD, 메타데이터, API, persistence, filtering |
| 예시 | HNSW, IVF, LSH 같은 인덱스 구조 | Pinecone, Qdrant, Milvus, Weaviate 등 |
| 비유 | 책 뒤의 색인 | 도서관 시스템 전체 |

중요한 포인트는, **Vector Index는 Vector Database 안의 한 구성 요소일 수 있다**는 점입니다.

실무에서는 보통 다음 요소가 함께 필요합니다.

- 벡터 자체 저장
- 문서 ID와 메타데이터 저장
- 빠른 근사 검색 인덱스
- 필터링과 하이브리드 질의
- 업데이트, 삭제, 버전 관리


## Vector Database Pipeline

PDF에서도 강조되는 큰 흐름은 아래와 같습니다.

```text
Source Data -> Preprocess -> Embedding Model -> Embedded Vectors -> Vector Database
                                                          -> Indexing
User Query -> Querying (Retrieving) -> Result -> Postprocess -> Application
```

이걸 RAG 관점으로 다시 풀면:

1. 문서를 수집하고 정제합니다.
2. 문서를 청크 단위로 나눕니다.
3. 각 청크를 임베딩으로 바꿉니다.
4. 임베딩과 메타데이터를 Vector DB에 저장합니다.
5. 질문이 들어오면 질문도 임베딩합니다.
6. 비슷한 청크를 검색합니다.
7. 결과를 필터링하거나 재정렬합니다.
8. 최종 문맥을 LLM에 넘깁니다.


## Indexing이 왜 필요한가

문서가 10개라면 모든 벡터를 다 비교해도 괜찮습니다. 하지만 수십만 개, 수백만 개가 되면 모든 벡터를 매번 비교하는 것은 너무 느립니다.

그래서 등장하는 개념이 `Indexing`입니다.

Indexing은 쉽게 말해:

- 나중에 더 빨리 찾기 위해
- 벡터들을 어떤 구조로 미리 정리해 두는 과정

여기서 중요한 개념이 `ANN(Approximate Nearest Neighbor)`입니다.

- `kNN`: 가장 가까운 이웃을 정확하게 찾는 방식
- `ANN`: 아주 약간의 정확도 손실을 감수하고 훨씬 빠르게 찾는 방식

실무에서는 보통 `정확도 100%`보다 `충분히 높은 정확도 + 빠른 응답`이 더 중요할 때가 많습니다.


## Indexing에서 보는 핵심 tradeoff

인덱스를 고를 때는 보통 세 가지를 같이 봅니다.

- `Recall`: 관련 문서를 얼마나 잘 놓치지 않는가
- `Latency`: 얼마나 빨리 검색되는가
- `Memory / Storage`: 인덱스가 얼마나 무거운가

즉, 인덱스는 만능이 아니라 tradeoff입니다.

- 더 빠르게 찾고 싶으면 근사 탐색을 더 많이 사용하게 됨
- 더 정확하게 찾고 싶으면 비용과 시간이 늘어날 수 있음
- 저장 공간을 줄이고 싶으면 압축을 쓰게 되지만 정보 손실이 생길 수 있음


## 대표적인 인덱스 구조

PDF에서는 Indexing 구조를 크게 다음처럼 소개합니다.

- `Hash Index`
- `Tree Index`
- `Inverted File Index (IVF)`
- `Graph Index`

또 별도로 저장 공간과 속도를 위한 압축 관점에서:

- `Flat`
- `Scalar Quantization`
- `Product Quantization`

이 구조를 외우는 것보다 중요한 것은, **왜 여러 구조가 존재하는지**를 이해하는 것입니다.


## Hash Index와 LSH

Hash Index는 비슷한 벡터가 비슷한 버킷으로 들어가도록 해 빠르게 후보를 줄이는 방식입니다.

여기서 자주 등장하는 것이 `LSH(Locality Sensitive Hashing)`입니다.

핵심 아이디어:

- 가까운 벡터는 같은 해시 버킷으로 갈 가능성을 높이고
- 먼 벡터는 다른 버킷으로 가게 만들어
- 전체 탐색 대신 일부 후보만 보게 함

장점:

- 후보 축소가 빠름
- 특정 데이터 분포에서 효율적일 수 있음

주의점:

- 버킷 설계가 어렵고
- 일반적인 최신 Vector DB 실무에서는 HNSW나 IVF 계열이 더 자주 거론되는 편입니다.


## Tree Index와 Annoy

Tree Index는 데이터를 계층적으로 나누어 내려가며 후보를 찾는 방식입니다.

PDF에서는 `Annoy`도 함께 등장합니다. Annoy는 Spotify에서 만든 ANN 라이브러리로, 여러 개의 랜덤 트리를 만들어 검색 후보를 좁히는 접근으로 많이 알려져 있습니다.

장점:

- 구조가 비교적 직관적임
- 특정 사용 사례에서 빠른 근사 검색 가능

한계:

- 고차원 벡터에서 항상 최선은 아님
- 업데이트 빈도, 데이터 분포, 정확도 요구에 따라 더 적합한 구조가 따로 있을 수 있음


## Inverted File Index (IVF)

IVF는 Vector DB에서 아주 자주 만나는 개념입니다.

아이디어는 단순합니다.

1. 전체 벡터를 몇 개의 대표 중심점(`centroid`) 기준으로 군집화합니다.
2. 질의가 들어오면 모든 벡터를 다 보지 않고, 질의와 가까운 몇 개의 군집만 먼저 봅니다.

즉, 전체 검색 공간을 `미리 나눠 두고`, 질의 때는 일부 구역만 탐색하는 방식입니다.

장점:

- 전체 탐색보다 훨씬 빠를 수 있음
- 큰 데이터셋에서 효율적

주의점:

- centroid 품질과 탐색할 리스트 수가 성능에 큰 영향을 줌
- 군집이 잘못 잡히면 관련 벡터를 놓칠 수 있음


## Graph Index와 HNSW

`HNSW(Hierarchical Navigable Small World)`는 현대 Vector DB에서 가장 자주 등장하는 인덱스 중 하나입니다.

핵심 아이디어:

- 벡터들을 그래프의 노드로 보고
- 가까운 벡터끼리 연결한 뒤
- 상위 레벨에서 거칠게 접근하고 하위 레벨에서 정밀하게 내려오며 탐색합니다.

장점:

- 높은 recall과 빠른 검색 속도의 균형이 좋음
- 실무에서 많이 채택됨

주의점:

- 메모리를 많이 사용할 수 있음
- 파라미터 튜닝에 따라 성능 차이가 큼

초심자 입장에서는 `HNSW = 많이 쓰는 대표 ANN 인덱스`라고 먼저 이해해도 충분합니다.


## Compression: Flat, Scalar Quantization, Product Quantization

벡터가 많아지면 검색 속도뿐 아니라 저장 공간도 문제가 됩니다. 그래서 압축이 등장합니다.

### 1. Flat

- 원래 벡터를 거의 그대로 저장
- 정확도는 좋지만 메모리 사용량이 큼

### 2. Scalar Quantization (SQ)

- 각 차원의 실수값을 더 작은 정수 표현으로 바꿈
- 예: `float32 -> int8`
- 메모리를 줄이지만 일부 정보 손실이 생김

### 3. Product Quantization (PQ)

- 긴 벡터를 여러 부분으로 쪼갠 뒤
- 각 부분을 대표 코드북으로 근사 표현
- 대용량 벡터 검색에서 매우 중요한 압축 기법

정리하면:

- `Flat`: 정확도 우선
- `SQ`: 단순하고 가벼운 압축
- `PQ`: 더 적극적인 압축과 근사 검색 최적화


In [1]:
import numpy as np

query = np.array([0.7, 0.1, 0.8, 0.2], dtype=float)
doc_a = np.array([0.68, 0.12, 0.77, 0.21], dtype=float)
doc_b = np.array([0.20, 0.80, 0.10, 0.90], dtype=float)

def euclidean(a, b):
    return np.linalg.norm(a - b)

def dot_product(a, b):
    return float(np.dot(a, b))

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for name, doc in [("doc_a", doc_a), ("doc_b", doc_b)]:
    print(name)
    print("  euclidean:", round(euclidean(query, doc), 4))
    print("  dot product:", round(dot_product(query, doc), 4))
    print("  cosine:", round(cosine_similarity(query, doc), 4))


doc_a
  euclidean: 0.0424
  dot product: 1.146
  cosine: 0.9996
doc_b
  euclidean: 1.3115
  dot product: 0.48
  cosine: 0.3608


## Querying이란 무엇인가

Indexing이 `미리 정리해 두는 단계`라면, Querying은 `질문이 들어왔을 때 실제로 찾는 단계`입니다.

PDF에서는 Querying 파트에서 다음 구분이 중요하게 다뤄집니다.

- `Keyword Search`
- `Attribute Filter`
- `Sparse Vector Search`
- `Semantic Search`
- `Hybrid Search`

이 구분은 RAG 성능을 좌우합니다. 왜냐하면 실제 사용자 질문은 항상 의미 검색만으로 해결되지 않기 때문입니다.


## Keyword Search, Attribute Filter, Sparse Search

### 1. Keyword Search

질문 속 단어와 문서 속 단어가 직접 맞는지 보는 방식입니다.

- 장점: 정확한 용어 매칭에 강함
- 단점: 표현이 조금만 달라져도 놓칠 수 있음

### 2. Attribute Filter

메타데이터를 기반으로 후보를 좁히는 방식입니다.

예:

- 부서가 `finance`인 문서만 검색
- 날짜가 `2025-01-01` 이후인 문서만 검색
- 권한이 있는 사용자에게만 특정 문서 노출

RAG 실무에서는 이 필터가 매우 중요합니다. 특히 사내 문서 검색에서는 보안과 권한 때문에 필수에 가깝습니다.

### 3. Sparse Vector Search

단어 단위 가중치를 가진 희소 벡터 기반 검색입니다.

대표 예:

- `BM25`
- `SPLADE`

Sparse Search는 정확한 용어 일치가 중요한 문제에서 Dense Search보다 더 강할 수 있습니다.


## BM25와 SPLADE를 왜 알아야 할까

### BM25

전통적인 정보 검색에서 가장 널리 쓰이는 scoring 방식 중 하나입니다.

- `TF(Term Frequency)`: 특정 단어가 문서에 얼마나 자주 나오나
- `IDF(Inverse Document Frequency)`: 전체 문서에서 그 단어가 얼마나 드문가

즉, 많이 나오고 동시에 희귀한 단어일수록 더 중요하게 봅니다.

### SPLADE

Sparse Search를 신경망 방식으로 강화한 접근이라고 이해하면 좋습니다.

- BERT 계열 표현을 활용해
- 단순 exact match를 넘어
- term expansion 성격을 가지는 sparse representation을 만듭니다.

초심자 관점의 핵심은 이겁니다.

- `Dense Search`: 의미 유사성에 강함
- `Sparse Search`: 정확한 용어 매칭과 키워드 신호에 강함


## Semantic Search와 Distance Metrics

Semantic Search는 질문과 문서의 **의미적 유사성**을 벡터 공간에서 비교하는 방식입니다.

여기서 자주 등장하는 거리/유사도 지표가 있습니다.

### 1. Euclidean Distance

- 벡터 사이의 물리적 거리
- 작을수록 가깝다고 해석

### 2. Dot Product

- 방향과 크기의 영향을 함께 받음
- 임베딩이 정규화되었는지 여부에 따라 해석이 달라질 수 있음

### 3. Cosine Similarity

- 방향 유사성에 집중
- 크기보다 각도 중심으로 비교
- 텍스트 임베딩 실습에서 가장 자주 접하는 지표 중 하나

실무에서는 임베딩 모델과 DB 설정에 따라 어떤 metric이 기본인지 반드시 확인해야 합니다.


In [2]:
query = np.array([1.0, 2.0, 3.0])
docs = {
    "policy_doc": np.array([1.1, 2.1, 2.9]),
    "irrelevant_doc": np.array([3.0, 0.5, 0.2]),
}

rows = []
for name, vec in docs.items():
    rows.append(
        {
            "doc": name,
            "euclidean": round(float(euclidean(query, vec)), 4),
            "dot": round(float(dot_product(query, vec)), 4),
            "cosine": round(float(cosine_similarity(query, vec)), 4),
        }
    )

rows


[{'doc': 'policy_doc', 'euclidean': 0.1732, 'dot': 14.0, 'cosine': 0.9989},
 {'doc': 'irrelevant_doc', 'euclidean': 3.7537, 'dot': 4.6, 'cosine': 0.4034}]

## Hybrid Search가 중요한 이유

현실의 질의는 보통 하나의 방식으로만 잘 해결되지 않습니다.

예를 들어:

- 제품 코드명, 에러 코드, 규정 번호는 키워드 매칭이 중요할 수 있음
- 사용자의 의도와 맥락은 의미 검색이 중요할 수 있음

그래서 등장하는 것이 `Hybrid Search`입니다.

Hybrid Search는 보통 다음을 결합합니다.

- Sparse Search 결과
- Dense Search 결과
- 경우에 따라 Metadata Filtering 결과

핵심은 `여러 검색 신호를 어떻게 합칠까` 입니다.


## 대표적인 Fusion 방식

PDF에서는 Hybrid Search 맥락에서 여러 fusion 아이디어가 소개됩니다.

### 1. Naive Weighted Score

- 예: `0.3 * sparse + 0.7 * dense`
- 단순하지만 스코어 스케일 차이를 잘 다뤄야 함

### 2. RRF(Reciprocal Rank Fusion)

- 점수 자체보다 순위를 결합하는 방식
- 서로 다른 검색기의 점수 체계가 달라도 비교적 안정적

### 3. RSF, DBSF 같은 score fusion 계열

- 점수를 정규화하거나 분포 기준으로 결합하는 접근
- 검색기별 스코어 분포 차이를 보정하려는 목적이 큼

초심자는 먼저 `가중합`과 `RRF`만 확실히 이해해도 충분합니다.


In [3]:
dense_ranking = ["doc_2", "doc_1", "doc_4", "doc_3"]
sparse_ranking = ["doc_1", "doc_3", "doc_2", "doc_5"]

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

rrf([dense_ranking, sparse_ranking])


[('doc_1', 0.03252247488101534),
 ('doc_2', 0.032266458495966696),
 ('doc_3', 0.031754032258064516),
 ('doc_4', 0.015873015873015872),
 ('doc_5', 0.015625)]

## Vector Database 제품을 볼 때의 기준

PDF에서는 여러 Vector DB가 예시로 등장합니다.

- Pinecone
- Qdrant
- Vald
- Weaviate
- Vespa
- Milvus
- LanceDB

중요한 것은 제품 이름보다 **비교 기준**입니다.

### 1. 운영 방식

- Managed Cloud 중심인가
- Self-hosted도 가능한가

### 2. 인덱스 지원

- HNSW를 쓰는가
- IVF/PQ 같은 압축 기반 검색을 지원하는가
- Dense만 강한가, Sparse/Hybrid도 강한가

### 3. 메타데이터와 필터링

- 권한, 날짜, 부서, 문서 타입 같은 필터가 쉬운가

### 4. 개발 경험

- Python SDK가 쉬운가
- 문서화가 잘 되어 있는가
- 로컬 실험과 운영 전환이 쉬운가

### 5. RAG 친화성

- Chunk metadata 관리가 편한가
- Hybrid Search가 되는가
- 업데이트와 삭제가 쉬운가


## Pinecone 예시로 보는 데이터 모델

PDF의 후반부에서는 Pinecone을 예시로 dense vector, sparse vector, metadata 개념을 설명합니다.

초심자가 여기서 꼭 잡아야 할 포인트는 아래입니다.

- `Record ID`: 문서 또는 청크를 구분하는 식별자
- `Dense Vector`: 의미 검색용 임베딩
- `Sparse Vector`: 키워드/하이브리드 검색용 신호
- `Metadata`: 필터링과 후처리에 쓰는 속성값

즉, 실제 Vector DB 레코드는 단순히 `숫자 배열 하나`가 아니라, 검색과 운영에 필요한 여러 정보를 함께 가지는 경우가 많습니다.


In [ ]:
example_record = {
    "id": "doc-12-chunk-03",
    "dense_vector": [0.12, 0.44, 0.91, 0.33],
    "sparse_vector": {"indices": [3, 18, 44], "values": [1.3, 0.7, 2.1]},
    "metadata": {
        "department": "finance",
        "source": "internal_policy",
        "page": 8,
        "security_level": "internal",
    },
}

example_record

{'id': 'doc-12-chunk-03',
 'dense_vector': [0.12, 0.44, 0.91, 0.33],
 'sparse_vector': {'indices': [3, 18, 44], 'values': [1.3, 0.7, 2.1]},
 'metadata': {'department': 'finance',
  'source': 'internal_policy',
  'page': 8,
  'security_level': 'internal'}}

## 2주차 체크리스트

아래 질문에 답할 수 있으면 이번 주 핵심은 잡힌 상태입니다.

- Vector Index와 Vector Database의 차이를 설명할 수 있는가
- ANN이 왜 필요한지 설명할 수 있는가
- IVF와 HNSW가 어떤 문제를 풀기 위해 등장하는지 말할 수 있는가
- Dense Search와 Sparse Search의 강점 차이를 설명할 수 있는가
- Hybrid Search가 왜 실무에서 중요한지 설명할 수 있는가
- 메타데이터 필터링이 보안형 RAG에서 왜 중요한지 설명할 수 있는가


## 스스로 점검할 질문

1. 문서가 많지 않을 때도 복잡한 ANN 인덱스가 꼭 필요할까
2. 제품 코드나 정책 번호가 중요한 질의에서 Dense Search만 쓰면 어떤 문제가 생길까
3. 사내 문서 검색에서 메타데이터 필터링 없이 검색하면 어떤 보안 문제가 생길까
4. 정확도보다 응답 속도가 더 중요한 상황에서는 어떤 인덱스를 선호하게 될까
5. Hybrid Search 결과를 합칠 때 단순 가중합과 RRF의 차이는 무엇일까


2주차에서 중요한 것은 제품 이름을 외우는 것이 아니라, **벡터 저장과 검색이 어떻게 설계되는지**를 이해하는 것입니다.

다음 단계에서는 아래 순서로 이어지면 좋습니다.

- 실제 임베딩 모델로 문서를 벡터화해 보기
- 간단한 Vector Store에 저장해 similarity search 실행해 보기
- chunking 방식에 따라 retrieval 결과가 어떻게 달라지는지 비교해 보기
- dense only, sparse only, hybrid search의 결과 차이를 체감해 보기

이 노트를 읽고 나면, 이제 `왜 Vector DB가 필요한가`는 이해한 상태입니다. 다음부터는 직접 작은 문서셋으로 실험하면서 감을 잡으면 됩니다.
